# VoxDoc - Google Colab Deployment

Deploy the VoxDoc Voice Symptom Intake & Documentation Assistant on Google Colab with GPU support.

## Prerequisites
- **Google Colab with GPU runtime** (Runtime → Change runtime type → T4 GPU)
- **Hugging Face token** with "Read" access ([get one here](https://huggingface.co/settings/tokens))
- Accept model terms for [google/medasr](https://huggingface.co/google/medasr) and [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
- **ngrok authtoken** ([get one here](https://dashboard.ngrok.com/get-started/your-authtoken))
- **Google Drive** (optional) — keeps model downloads between Colab runtimes (Step 4)

## What you get
- MedGemma SOAP documentation generation on a free T4 GPU
- MedASR + Whisper speech-to-text
- Medical NER (condition & medication extraction)
- Public URL via ngrok tunnel
- A Verify step that checks speech-to-text accuracy after every change (Step 6)

## Step 1: Check GPU & Clone Repository

`BRANCH` picks which version of the code to run. `colab-gpu-testing` is the known-good version; set it to a feature branch (for example `stage-0-colab-reliability`) to try new work without touching it.

In [ ]:
import torch, subprocess

# Which version of the code to run. "colab-gpu-testing" is the known-good version;
# change this to a feature branch to try new work.
BRANCH = "colab-gpu-testing"
REPO_URL = "https://github.com/JoelJohnsonThomas/voice-symptom-documentation-assistant.git"
REPO_DIR = "/content/voice-symptom-documentation-assistant"

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB VRAM)")
else:
    print("WARNING: No GPU detected!")
    print("Go to Runtime -> Change runtime type -> Select 'T4 GPU'")
    print("Then re-run this cell.")

# Clone repository
!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"\nRepo cloned (branch: {BRANCH}, commit: {commit}). Working directory: {REPO_DIR}")

## Step 2: Install All Dependencies

This takes ~3-5 minutes. Every package the app needs is installed here — no manual pip installs later.

In [ ]:
import os, subprocess, sys
import importlib.metadata as metadata

# Make sure a GPU is attached before spending minutes installing.
# CPU-only runtimes don't have nvidia-smi at all, so a missing command also means "no GPU".
def gpu_name():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            return r.stdout.strip()
    except FileNotFoundError:
        pass
    return "NVIDIA GPU" if os.path.exists("/dev/nvidia0") else None

gpu = gpu_name()
if gpu is None:
    raise SystemExit("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then start again from Step 1. "
                     "If Colab says you've reached usage limits, try again later.")
print(f"GPU found: {gpu}")

# Lock Colab's own PyTorch build: every install below must keep these exact versions,
# so a package that wants a different torch fails visibly instead of breaking the GPU.
locked = {}
for name in ("torch", "torchvision", "torchaudio"):
    try:
        locked[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass
CONSTRAINTS = "/content/torch-constraints.txt"
with open(CONSTRAINTS, "w") as f:
    f.writelines(f"{name}=={version}\n" for name, version in locked.items())
print("Keeping Colab's PyTorch: " + ", ".join(f"{n} {v}" for n, v in locked.items()) + "\n")

def pip(*args):
    """Install packages and show result."""
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "-c", CONSTRAINTS, *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  FAILED: pip install {' '.join(args)}")
        if r.stderr:
            print(r.stderr[-1500:])
        return False
    return True

print("1/8  Transformers (MedGemma-compatible commit)...")
pip("git+https://github.com/huggingface/transformers.git@65dc261512cbdb1ee72b88ae5b222f2605aad8e5")

print("2/8  Web framework & networking...")
pip("fastapi", "uvicorn[standard]", "python-multipart",
    "websockets", "httpx")

print("3/8  ML & audio...")
pip("accelerate", "huggingface_hub>=0.24.0")
pip("librosa", "soundfile", "audioread", "noisereduce")
pip("openai-whisper", "--no-deps")  # --no-deps keeps Colab's torch untouched
pip("more-itertools", "tiktoken", "numba")  # whisper's actual deps, without torch
pip("faster-whisper")
pip("bitsandbytes")  # optional quantization support

print("4/8  Config & database...")
pip("pydantic", "pydantic-settings", "python-dotenv")
pip("sqlalchemy[asyncio]", "aiosqlite", "alembic")

print("5/8  Auth & security...")
pip("PyJWT", "passlib[bcrypt]", "pyotp",
    "qrcode[pil]", "authlib")
pip("cryptography")

print("6/8  Medical NLP (spaCy + optional SciSpaCy NER models)...")
pip("spacy", "--prefer-binary")
# scispacy pins an old spaCy with no wheels for Colab's Python, so skip the package.
# Models install with --no-deps; NER is optional and the app runs without it.
for url in ["https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz",
            "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz"]:
    if not pip(url, "--no-deps"):
        print("  (optional) NER model skipped — app will run without medical NER")
pip("presidio-analyzer", "presidio-anonymizer")

print("7/8  RAG & knowledge base...")
pip("chromadb", "sentence-transformers")

print("8/8  Infrastructure & utilities...")
pip("pyngrok", "nest-asyncio")

# Check CUDA in a fresh process — this kernel may hold a stale torch import.
CUDA_CHECK = ("import torch; ok = torch.cuda.is_available(); "
              "print(f'torch {torch.__version__}, CUDA {torch.version.cuda}, GPU usable: {ok}'); "
              "raise SystemExit(0 if ok else 1)")
def cuda_ok():
    r = subprocess.run([sys.executable, "-c", CUDA_CHECK], capture_output=True, text=True)
    lines = (r.stdout or r.stderr).strip().splitlines()
    return r.returncode == 0, (lines[-1] if lines else "no output")

ok, info = cuda_ok()
if not ok:
    print(f"\n  torch can't use the GPU ({info}).")
    print("  Reinstalling Colab's original PyTorch build...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *locked])
    pins = [f"{n}=={v}" for n, v in locked.items()]
    cuda_tag = locked.get("torch", "").partition("+")[2]  # e.g. "cu126"
    if cuda_tag:
        pip(*pins, "--index-url", f"https://download.pytorch.org/whl/{cuda_tag}")
    else:
        pip(*pins)
    ok, info = cuda_ok()
print(f"\nGPU torch: {'OK' if ok else 'BROKEN'} ({info})")
if not ok:
    print("  The server can't run without a working GPU. Use Runtime -> Disconnect and delete runtime,")
    print("  then start again from Step 1.")

# Verify critical imports (fresh process, so results reflect what uvicorn will see)
print("\nVerifying critical imports...")
mods = ["fastapi", "uvicorn", "torch", "transformers", "whisper", "librosa", "sqlalchemy",
        "aiosqlite", "alembic", "jwt", "passlib", "pyotp", "qrcode", "authlib", "cryptography",
        "spacy", "chromadb", "sentence_transformers", "httpx", "pyngrok", "presidio_analyzer",
        "soundfile", "pydantic", "pydantic_settings"]
check = "import importlib,sys\nbad=[]\nfor m in sys.argv[1:]:\n    try: importlib.import_module(m)\n    except Exception as e: bad.append(f'{m}: {e}')\nprint('\\n'.join(bad))"
r = subprocess.run([sys.executable, "-c", check, *mods], capture_output=True, text=True)
missing = r.stdout.strip()
if missing:
    print("Some packages failed to import:\n" + missing)
else:
    print("All packages installed successfully!")
print("\nIMPORTANT: Runtime -> Restart session now, then continue with Step 3.")

## Step 3: Configure Environment

Replace `HF_TOKEN` and `NGROK_AUTH_TOKEN` with your own values.

In [ ]:
import os
os.chdir("/content/voice-symptom-documentation-assistant")

# ──────────────────────────────────────────────
#  REPLACE THESE TWO VALUES WITH YOUR OWN
# ──────────────────────────────────────────────
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
# ──────────────────────────────────────────────

# ──────────────────────────────────────────────
#  FEATURE SWITCHES
#  Each new feature gets one line here and starts as False.
#  Turn one on only after the Verify step (Step 6) shows it helps.
# ──────────────────────────────────────────────
FEATURE_FLAGS = {
    # Stage 1 features will be added here, e.g.:
    # "MEDASR_LM_DECODING": False,
}

# The app refuses to start if .env has a name it doesn't know, so check the
# switches exist in this branch's app/config.py before writing them.
import re
config_src = open("app/config.py").read()
unknown = [k for k in FEATURE_FLAGS
           if not re.search(rf"^\s+{k.lower()}\s*:", config_src, re.M)]
if unknown:
    raise ValueError(f"These feature switches don't exist on this branch: {unknown}. "
                     "Remove them, or set BRANCH in Step 1 to the branch that adds them.")

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

    # Models — whisper-small for high accuracy speech recognition
    f.write("MEDASR_MODEL=google/medasr\n")
    f.write("MEDGEMMA_MODEL=google/medgemma-1.5-4b-it\n")
    f.write("WHISPER_MODEL=openai/whisper-small\n")
    f.write("MODEL_CACHE_DIR=./models\n")

    # GPU — MedGemma runs locally on T4
    f.write("DEVICE=cuda\n")
    f.write("ENABLE_GPU=true\n")
    f.write("COLAB_MODE=true\n")

    # MedGemma provider — local means load on this GPU
    f.write("MEDGEMMA_PROVIDER=local\n")
    f.write("HF_INFERENCE_PROVIDER=\n")
    f.write("HF_INFERENCE_TIMEOUT=120\n")

    # Generation parameters
    f.write("MEDGEMMA_MAX_TOKENS=1024\n")
    f.write("MEDGEMMA_REPETITION_PENALTY=1.1\n")

    # Compliance — REQUIRED or MedGemma won't load
    f.write("MEDGEMMA_TERMS_ACKNOWLEDGED=true\n")
    f.write("ENFORCE_MEDGEMMA_TERMS_ACKNOWLEDGEMENT=true\n")
    f.write("ALLOW_PHI_LOGGING=false\n")
    f.write("ENABLE_PHI_PERSISTENCE=false\n")
    f.write("AUDIT_LOGGING_ENABLED=true\n")

    # Audio — 2s streaming interval (fast on GPU)
    f.write("AUDIO_SAMPLE_RATE=16000\n")
    f.write("MAX_AUDIO_DURATION_SECONDS=300\n")
    f.write("STREAMING_INTERVAL_SECONDS=2.0\n")

    # Rate limiting
    f.write("RATE_LIMITING_ENABLED=true\n")
    f.write("RATE_LIMIT_GENERAL_RPM=60\n")
    f.write("RATE_LIMIT_INFERENCE_RPM=10\n")
    f.write("QUEUE_MAX_CONCURRENT_INFERENCES=2\n")
    f.write("QUEUE_MAX_SIZE=20\n")
    f.write("QUEUE_TIMEOUT_SECONDS=120\n")
    f.write("QUEUE_ESTIMATED_INFERENCE_SECONDS=10\n")

    # Monitoring
    f.write("METRICS_ENABLED=true\n")
    f.write("STRUCTURED_LOGGING_ENABLED=true\n")
    f.write("METRICS_ENDPOINT_AUTH_REQUIRED=false\n")

    # Security (dev defaults — not for production)
    f.write("ENCRYPTION_AT_REST_ENABLED=false\n")
    f.write("ENCRYPTION_MASTER_KEY=CHANGE_ME_IN_PRODUCTION\n")
    f.write("JWT_SECRET_KEY=CHANGE_ME_IN_PRODUCTION\n")

    # RAG (disabled for faster startup)
    f.write("RAG_ENABLED=false\n")
    f.write("RAG_EMBEDDING_MODEL=NeuML/pubmedbert-base-embeddings\n")

    # Logging
    f.write("LOG_LEVEL=INFO\n")

    # Feature switches (see FEATURE_FLAGS above)
    for name, enabled in FEATURE_FLAGS.items():
        f.write(f"{name}={'true' if enabled else 'false'}\n")

# Configure ngrok
from pyngrok import ngrok
import nest_asyncio
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
nest_asyncio.apply()

print("Environment configured!")
print(f"  Whisper model: openai/whisper-small (high accuracy)")
print(f"  MedGemma: local GPU inference (google/medgemma-1.5-4b-it)")
print(f"  MEDGEMMA_TERMS_ACKNOWLEDGED: true")
print(f"  Streaming interval: 2.0s (fast on GPU)")
print(f"  ngrok: ready")
print(f"  Feature switches: {FEATURE_FLAGS or 'none yet'}")

## Step 4: Keep Model Downloads on Google Drive (optional)

Each new Colab runtime starts with an empty disk, so the models (~10 GB, mostly MedGemma) download again. With `USE_DRIVE_CACHE = True` they are kept on your Google Drive instead.

- Needs about **12 GB free** on Drive.
- The first run still downloads everything. After that, models load from Drive.
- Reading from Drive can be slower than a fresh download. Step 5 prints how long startup took, so compare once with `True` and once with `False` and keep the faster one.
- Only model files go to Drive, never patient data.

In [ ]:
import os, shutil

# Leave False to keep the current behaviour (models download to local disk each new runtime).
USE_DRIVE_CACHE = False

CACHE_ROOT = "/content/drive/MyDrive/voxdoc/cache"
CACHE_VARS = ("HF_HOME", "TORCH_HOME", "HF_XET_CACHE", "HF_HUB_DISABLE_SYMLINKS_WARNING")

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CACHE_ROOT, exist_ok=True)

    try:
        free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1024**3
        if free_gb < 12:
            print(f"WARNING: Drive reports {free_gb:.1f} GB free; the models need about 12 GB.")
    except OSError:
        pass

    # These are environment variables, not .env lines: the app's config rejects unknown .env names.
    os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"   # MedASR, MedGemma, faster-whisper, embeddings
    os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"       # Silero VAD (torch.hub)
    os.environ["HF_XET_CACHE"] = "/content/hf_xet_cache"   # temporary download chunks stay on local disk
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"    # Drive has no symlinks; huggingface_hub copies instead
    print(f"Model cache: Google Drive ({CACHE_ROOT})")
else:
    for var in CACHE_VARS:
        os.environ.pop(var, None)
    print("Model cache: local disk (models download again on each new runtime).")
    print("Set USE_DRIVE_CACHE = True to keep them on Google Drive.")

## Step 5: Start the Server

This will:
1. Stop any server left over from an earlier run
2. Create a public ngrok URL
3. Start the FastAPI server **in the background** and show its log while the models load (MedASR ~30 seconds, MedGemma ~2-3 minutes; the first run downloads ~8 GB)
4. Print the public URL once the server is ready

The server keeps running after this cell finishes, so you can run Step 6 (Verify). Use the cells at the bottom to watch the log or stop the server.

In [ ]:
import os, sys, time, subprocess, requests
from pyngrok import ngrok

REPO_DIR = "/content/voice-symptom-documentation-assistant"
LOG_FILE = "/content/voxdoc_server.log"
os.chdir(REPO_DIR)
os.environ["MEDGEMMA_TERMS_ACKNOWLEDGED"] = "true"
os.environ["PYTHONUNBUFFERED"] = "1"  # server log lines appear immediately

# Without a usable GPU, MedGemma loads on the CPU in float32 (~17 GB per model) and Colab
# kills the server for running out of memory. Check first and stop with a clear message.
gpu_ok = subprocess.run(
    [sys.executable, "-c", "import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)"],
    capture_output=True,
).returncode == 0
if not gpu_ok:
    raise SystemExit("PyTorch can't use the GPU, so the server would run out of memory and be killed. "
                     "Check that Step 2 ended with 'GPU torch: OK', or run !nvidia-smi.")

# Stop a server or tunnel left over from an earlier run, so port 8000 is free.
ngrok.kill()
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], capture_output=True)
time.sleep(2)

PUBLIC_URL = ngrok.connect(8000).public_url

# Start the server in the background so other cells (Verify) can run while it's up.
server_log = open(LOG_FILE, "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR, stdout=server_log, stderr=subprocess.STDOUT,
)
print(f"Server starting (full log: {LOG_FILE}). Loading models takes a few minutes...\n")

# Show the log while waiting. /api/health only answers once all models have loaded.
start, pos, ready = time.time(), 0, False
while time.time() - start < 20 * 60:
    with open(LOG_FILE) as f:
        f.seek(pos)
        new_text = f.read()
        pos = f.tell()
    if new_text:
        print(new_text, end="")
    if server.poll() is not None:
        break
    try:
        ready = requests.get("http://127.0.0.1:8000/api/health", timeout=3).ok
    except requests.RequestException:
        ready = False
    if ready:
        break
    time.sleep(3)

react_built = os.path.exists(f"{REPO_DIR}/frontend/dist/index.html")
if ready:
    print(f"\n{'='*60}")
    print(f"  PUBLIC URL: {PUBLIC_URL}")
    print(f"  Ready after {time.time() - start:.0f}s")
    print(f"  UI: {'React frontend' if react_built else 'Vanilla JS fallback (React build not found)'}")
    print(f"{'='*60}")
    print("  Open the URL in your browser. Next: run Step 6 (Verify).")
elif server.poll() is not None:
    print(f"\nServer stopped (exit code {server.returncode}). The error is in the log above.")
    if server.returncode == -9:
        print("Exit code -9 means the system killed it, almost always for running out of memory (RAM).")
        print("If the log shows 'Using float32 precision on CPU', PyTorch couldn't use the GPU.")
else:
    print("\nServer still not ready after 20 minutes. Check the log above or run the 'Server log' cell.")

## Step 6: Verify Speech-to-Text

Run this after Step 5 says the server is ready (about 1 minute). It shows:
- which speech models loaded (MedASR, or the Whisper fallback)
- GPU memory in use
- the transcript of MedASR's sample clip (a CT chest dictation) and its **word error rate** (the share of words that are wrong; lower is better)

**Your own recordings:** put `name.wav` plus `name.txt` (the exact words spoken) in `MyDrive/voxdoc/verify/` (needs Step 4) or `/content/verify_clips/`. Supported formats: wav, flac, ogg, mp3, webm. Use recordings without real patient information.

Each run is added to `verify_history.csv` (on Drive when it's mounted), so you can compare branches before turning a feature switch on.

In [ ]:
import os, re, csv, glob, time, subprocess, requests
from datetime import datetime
from huggingface_hub import hf_hub_download

API = "http://127.0.0.1:8000"
REPO_DIR = "/content/voice-symptom-documentation-assistant"
CLIP_DIRS = ["/content/drive/MyDrive/voxdoc/verify", "/content/verify_clips"]
AUDIO_EXTS = (".wav", ".flac", ".ogg", ".mp3", ".webm")

# Correct transcript of MedASR's sample clip (from Google's MedASR quick-start notebook).
SAMPLE_REFERENCE = (
    "Exam type CT chest PE protocol period. Indication 54 year old female, shortness of breath, "
    "evaluate for PE period. Technique standard protocol period. Findings colon. Pulmonary vasculature "
    "colon. The main PA is patent period. There are filling defects in the segmental branches of the "
    "right lower lobe comma compatible with acute PE period. No saddle embolus period. Lungs colon. "
    "No pneumothorax period. Small bilateral effusions comma right greater than left period. "
    "New paragraph. Impression colon Acute segmental PE right lower lobe period."
)

def words(text):
    """Lowercase, drop punctuation and spoken formatting words, so MedASR and Whisper are scored fairly."""
    text = re.sub(r"[^a-z0-9' ]", " ", text.lower().replace("</s>", " "))
    text = re.sub(r"\bnew paragraph\b", " ", text)
    text = re.sub(r"\b(period|comma|colon)\b", " ", text)
    return text.split()

def word_error_rate(reference, hypothesis):
    ref, hyp = words(reference), words(hypothesis)
    row = list(range(len(hyp) + 1))          # word-level edit distance, one row at a time
    for i in range(1, len(ref) + 1):
        diag, row[0] = row[0], i
        for j in range(1, len(hyp) + 1):
            above = row[j]
            row[j] = min(above + 1, row[j - 1] + 1, diag + (ref[i - 1] != hyp[j - 1]))
            diag = above
    return row[len(hyp)] / max(len(ref), 1)

def transcribe(path):
    for _ in range(4):
        with open(path, "rb") as f:
            r = requests.post(f"{API}/api/transcribe",
                              files={"audio": (os.path.basename(path), f)}, timeout=600)
        if r.status_code == 429:              # inference rate limit (10/min): wait and retry
            time.sleep(15)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True).stdout.strip()

# ── 1. Server status ─────────────────────────────────────────────
try:
    health = requests.get(f"{API}/api/health", timeout=10).json()
except requests.RequestException:
    raise SystemExit("Server isn't answering. Run Step 5 first and wait for 'PUBLIC URL'.")

branch, commit = git("rev-parse", "--abbrev-ref", "HEAD"), git("rev-parse", "--short", "HEAD")
backends = health.get("asr_backends")         # older branches don't report this
print(f"Code:            {branch} @ {commit}")
print(f"Server:          {health.get('status')}")
if backends is not None:
    loaded = [name for name, ok in backends.items() if ok]
    print(f"Speech models:   {', '.join(loaded) or 'none'}")
    if not backends.get("medasr"):
        print("  WARNING: MedASR didn't load, so Whisper is doing all transcription.")
        print("  Check your HF token and that you accepted the google/medasr terms.")
print(f"MedGemma ready:  {health.get('medgemma_ready')}")

try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True)
except FileNotFoundError:  # CPU-only runtime
    smi = None
if smi and smi.returncode == 0 and smi.stdout.strip():
    name, used, total = [x.strip() for x in smi.stdout.splitlines()[0].split(",")]
    print(f"GPU memory:      {int(used)/1024:.1f} of {int(total)/1024:.1f} GB used ({name})")
print(f"Model cache:     {os.environ.get('HF_HOME', 'local disk')}")

# ── 2. Collect clips: MedASR's sample + your own ─────────────────
env = {}
if os.path.exists(f"{REPO_DIR}/.env"):
    for line in open(f"{REPO_DIR}/.env"):
        if "=" in line and not line.startswith("#"):
            key, value = line.strip().split("=", 1)
            env[key] = value

clips = [("medasr_sample",
          hf_hub_download("google/medasr", "test_audio.wav", token=env.get("HF_TOKEN") or None),
          SAMPLE_REFERENCE)]
for folder in CLIP_DIRS:
    for path in sorted(glob.glob(f"{folder}/*")):
        stem, ext = os.path.splitext(path)
        if ext.lower() in AUDIO_EXTS and os.path.exists(stem + ".txt"):
            clips.append((os.path.basename(path), path, open(stem + ".txt").read()))

# ── 3. Transcribe and score ──────────────────────────────────────
print(f"\nTranscribing {len(clips)} clip(s)...\n")
results = []
for name, path, reference in clips:
    t0 = time.time()
    try:
        out = transcribe(path)
    except Exception as e:
        print(f"{name}: FAILED ({e})")
        continue
    results.append({
        "time": datetime.now().isoformat(timespec="seconds"), "branch": branch, "commit": commit,
        "clip": name, "wer_percent": round(100 * word_error_rate(reference, out["transcript"]), 1),
        "audio_seconds": round(out.get("duration_seconds", 0), 1),
        "response_seconds": round(time.time() - t0, 1),
        "language": out.get("detected_language"),
        "speech_models": "+".join(k for k, v in (backends or {}).items() if v),
    })
    if name == "medasr_sample":
        print(f"Sample transcript: {out['transcript'][:400]}\n")

print(f"{'Clip':<28}{'Word errors':>12}{'Audio':>9}{'Took':>8}  Lang")
for r in results:
    print(f"{r['clip'][:27]:<28}{r['wer_percent']:>11.1f}%{r['audio_seconds']:>8.1f}s"
          f"{r['response_seconds']:>7.1f}s  {r['language']}")
if len(clips) == 1:
    print("\nThe sample is a radiology dictation, so it mostly measures MedASR. "
          "Add your own clips (see the note above) for a real-world score.")

# ── 4. Append to history so branches can be compared ─────────────
history = ("/content/drive/MyDrive/voxdoc/verify_history.csv"
           if os.path.isdir("/content/drive/MyDrive") else "/content/verify_history.csv")
if results:
    os.makedirs(os.path.dirname(history), exist_ok=True)
    is_new = not os.path.exists(history)
    with open(history, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        if is_new:
            writer.writeheader()
        writer.writerows(results)
    print(f"\nSaved to {history}")

## Server Log & Stop

**Server log** shows the log live. Stopping that cell does *not* stop the server.

**Stop server** shuts down the server and the ngrok tunnel.

In [ ]:
# Server log (live). Stop this cell any time; the server keeps running.
import time
with open("/content/voxdoc_server.log") as f:
    print(f.read()[-5000:], end="")  # the most recent part of the log
    try:
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(1)
    except KeyboardInterrupt:
        print("\n(Stopped showing the log. The server is still running.)")

In [ ]:
# Stop server
import subprocess
from pyngrok import ngrok
subprocess.run(["pkill", "-f", "uvicorn app.main:app"])
ngrok.kill()
print("Server and ngrok tunnel stopped.")

## Usage & Troubleshooting

### How to use
1. Copy the ngrok URL printed above and open it in your browser
2. Click **Start Recording** or upload an audio file
3. The app transcribes speech and generates SOAP documentation using MedGemma

### To stop
- Run the **Stop server** cell (stopping the Step 5 cell no longer stops the server)

### Common issues

| Problem | Fix |
|---------|-----|
| `No GPU detected` | Runtime → Change runtime type → T4 GPU, then restart |
| `ModuleNotFoundError` | Re-run the Step 2 install cell |
| `Skipping MedGemma preload: terms not acknowledged` | Make sure `.env` has `MEDGEMMA_TERMS_ACKNOWLEDGED=true` |
| `torch.cuda.is_available() is False` | CUDA torch got overwritten — re-run Step 2 |
| ngrok `ERR_NGROK_108` | Free tier allows 1 tunnel — stop other Colab sessions |
| `en_ner_bc5cdr_md not found` | Re-run Step 2 (installs SciSpaCy models) |
| Documentation shows "N/A" | Model may need warmup — try generating again |
| `Address already in use` | Run Step 5 again (it stops the old server first) or run **Stop server** |
| `These feature switches don't exist on this branch` | Remove that switch from `FEATURE_FLAGS`, or set `BRANCH` to the branch that adds it |
| Verify says MedASR didn't load | Check `HF_TOKEN` and accept the [google/medasr](https://huggingface.co/google/medasr) terms |
| `No GPU attached` (Step 2) | Runtime → Change runtime type → T4 GPU. If Colab mentions usage limits, try again later |
| `Server stopped (exit code -9)` | Out of memory — usually PyTorch can't use the GPU, so models loaded on the CPU. Make sure Step 2 ends with `GPU torch: OK`; if not, Runtime → Disconnect and delete runtime and start again |

### Notes
- Free Colab sessions last up to 12 hours
- The ngrok URL changes each restart
- Models download again on each new runtime, unless Step 4 (Google Drive cache) is on
- GPU inference is ~10x faster than CPU